# SCM — RTL-accurate Python reference model

`scm/src/rtl/` 의 SystemVerilog 를 **직접 분석해서** 파이썬으로 재현한 레퍼런스 모델입니다.
각 RTL 모듈과 1:1 로 매칭되게 작성했고, 셀 단위로 실행하며 디버깅 / 시각화합니다.

### RTL 블록 매핑
| Cell 섹션 | RTL 모듈 | 파일 |
|---|---|---|
| 2 | `qpsk_modulator` | `qpsk_modulator.sv` |
| 3 | `frame_builder` FSM | `frame_builder.sv` |
| 4 | `axis_upsample_zeros` | `axis_upsample_zeros.sv` |
| 5 | TX `fir_rrc` | Xilinx FIR IP (`fir_rrc`) |
| 6 | channel | AWGN (RTL 없음) |
| 7 | RX `fir_rrc_rx` | Xilinx FIR IP (`fir_rrc_rx`) |
| 8 | `axis_downsample_pick` | `axis_downsample_pick.sv` |
| 9 | `preamble_correlator` | `preamble_correlator.sv` |
| 10 | `frame_sync_detector` | `frame_sync_detector.sv` |
| 11 | `qpsk_demodulator` | `qpsk_demodulator.sv` |

### RTL 파라미터 (`qpsk_frame_sync_top`)
`W=16, W_2=40, W_3=56, SPS=4, OFFSET=0, PREAMBLE_LEN=16, W_CORR=72, SYNC_THRESHOLD=5e8`

### 주의 — RTL 특이 동작
- `frame_sync_detector` 는 `mag_sq[127:64] > THRESHOLD` — **상위 64비트로 비교**. 이 모델도 그대로 재현.
- `axis_downsample_pick OFFSET=0` (top-level). 파이썬에선 `offset=0` 으로 동일.
- `preamble_correlator` shift reg 는 `shift[0]=newest` → 계수는 `PREAMBLE[L-1-k]` 로 time-reverse.
- `qpsk_demodulator` 는 sign bit (MSB) 로 결정 — 0 은 bit 0 (RTL 상 `I>=0 → 0`).


In [ ]:
import numpy as np
import matplotlib.pyplot as plt
%matplotlib inline

np.set_printoptions(linewidth=120, suppress=True)

# =============================================================================
# RTL top-level parameters (from qpsk_frame_sync_top.sv)
# =============================================================================
W             = 16          # symbol bit width
W_2           = 40          # TX RRC output width
W_3           = 56          # RX RRC output width
SPS           = 4           # samples per symbol
OFFSET        = 0           # downsample pick offset (top-level)
PREAMBLE_LEN  = 16
W_CORR        = 72
SCALE         = 12000       # preamble / modulator amplitude
SYNC_THRESHOLD = 500_000_000  # default in top (adjust for SNR)

COOLDOWN_CYCLES = 32        # frame_sync_detector
print(f"Config: W={W}, SPS={SPS}, PREAMBLE_LEN={PREAMBLE_LEN}, SCALE={SCALE}")


## 1. Preamble & RRC Constants

Preamble 패턴은 `frame_builder.sv` / `preamble_correlator.sv` 의 `localparam` 과 **완전히 일치**해야 합니다.
RRC 계수는 `src/coe/rrc_sps4_beta0p35_span10_q16p.coe` 와 같은 16-bit 양자화 값.


In [ ]:
# Preamble (exact copy of frame_builder.sv PREAMBLE_I / PREAMBLE_Q)
PREAMBLE_I = np.array([
     SCALE,  SCALE,  SCALE, -SCALE,
    -SCALE,  SCALE, -SCALE,  SCALE,
     SCALE, -SCALE, -SCALE,  SCALE,
    -SCALE, -SCALE,  SCALE,  SCALE,
], dtype=np.int64)

PREAMBLE_Q = np.array([
     SCALE,  SCALE, -SCALE, -SCALE,
     SCALE,  SCALE, -SCALE, -SCALE,
     SCALE, -SCALE, -SCALE,  SCALE,
     SCALE,  SCALE, -SCALE,  SCALE,
], dtype=np.int64)

# RRC 16-bit quantized coefficients (from COE)
RRC_COEFS_INT = np.array([
    224, -71, -348, -307, 61, 398, 286, -286, -761, -442,
    766, 1954, 1708, -660, -4042, -5641, -2533, 6187, 18177, 28625,
    32767,
    28625, 18177, 6187, -2533, -5641, -4042, -660, 1708, 1954, 766,
    -442, -761, -286, 286, 398, 61, -307, -348, -71, 224,
], dtype=np.int64)

RRC_DELAY = (len(RRC_COEFS_INT) - 1) // 2   # = 20 samples (single filter)
print(f"RRC taps={len(RRC_COEFS_INT)}, single-filter delay={RRC_DELAY} samples")

fig, axes = plt.subplots(1, 3, figsize=(15, 3))
axes[0].stem(RRC_COEFS_INT); axes[0].set_title("RRC coefs (Q16 int)"); axes[0].grid(True)
axes[1].stem(PREAMBLE_I, linefmt="C0-", markerfmt="C0o", label="I")
axes[1].stem(PREAMBLE_Q, linefmt="C1-", markerfmt="C1x", label="Q")
axes[1].set_title("Preamble (16 symbols)"); axes[1].legend(); axes[1].grid(True)
axes[2].scatter(PREAMBLE_I, PREAMBLE_Q, s=50)
axes[2].set_title("Preamble constellation"); axes[2].axis("equal"); axes[2].grid(True)
plt.tight_layout(); plt.show()


## 2. `qpsk_modulator` (bits → symbols)

```verilog
mapped_i = in_data[0] ? -SCALE : +SCALE;
mapped_q = in_data[1] ? -SCALE : +SCALE;
```
2-bit 입력 한 개당 I/Q 한 심볼. **bit 0 이 I, bit 1 이 Q.**


In [ ]:
def qpsk_modulator(bits_pairs):
    """
    bits_pairs : (N,) int8 array, each element in {0,1,2,3}, representing
                 {Q, I} bit pair — i.e. bits_pairs[n] = (Q<<1) | I
    Matches `in_data[1:0]` semantics of qpsk_modulator.sv.
    Returns (sym_i, sym_q) int64 arrays.
    """
    bi = (bits_pairs & 0x1).astype(np.int8)   # in_data[0]
    bq = ((bits_pairs >> 1) & 0x1).astype(np.int8)  # in_data[1]
    sym_i = np.where(bi == 1, -SCALE, SCALE).astype(np.int64)
    sym_q = np.where(bq == 1, -SCALE, SCALE).astype(np.int64)
    return sym_i, sym_q


def bits_to_pairs(bits):
    """Pack a flat bit stream (MSB=Q, LSB=I) into 2-bit values."""
    assert len(bits) % 2 == 0
    bi = bits[0::2] & 0x1          # I bits
    bq = bits[1::2] & 0x1          # Q bits
    return ((bq << 1) | bi).astype(np.int8)


## 3. `frame_builder` (FSM: IDLE → PREAMBLE → PAYLOAD → IDLE)

RTL 특징:
- PREAMBLE 상태에선 `in_ready=0` (입력 소비 안 함), preamble 16 심볼을 직접 출력
- PAYLOAD 상태 진입 후 payload_len 심볼을 pass-through
- 우리는 back-pressure 없는 쭉 이어지는 벡터 연산으로 단순화 (out_ready=1 가정)


In [ ]:
def frame_builder(payload_i, payload_q):
    """Preamble (16) + payload, matching frame_builder.sv output order."""
    frame_i = np.concatenate([PREAMBLE_I, payload_i])
    frame_q = np.concatenate([PREAMBLE_Q, payload_q])
    return frame_i, frame_q


## 4. `axis_upsample_zeros` (1× → SPS×, zero insertion)

RTL:
- `phase==0` 에서 `hold_i/hold_q` 출력, `phase==1..SPS-1` 은 `'0`
- active flag + back-pressure 로 심볼당 정확히 SPS 샘플 출력

벡터화: `y[n*SPS] = x[n]`, 나머지는 0.


In [ ]:
def axis_upsample_zeros(sym_i, sym_q, sps=SPS):
    n = len(sym_i)
    up_i = np.zeros(n * sps, dtype=np.int64)
    up_q = np.zeros(n * sps, dtype=np.int64)
    up_i[::sps] = sym_i
    up_q[::sps] = sym_q
    return up_i, up_q


## 5. TX RRC filter (`fir_rrc` IP)

Xilinx FIR Compiler = `np.convolve(x, h, mode='full')` 와 동일.
TX 측은 16-bit coef × 16-bit data = W_2=40-bit 출력.


In [ ]:
def fir_rrc(x, coefs=RRC_COEFS_INT):
    """Bit-accurate integer convolution (matches Vivado FIR Compiler 'Full Precision')."""
    return np.convolve(x.astype(np.int64), coefs, mode="full").astype(np.int64)


## 6. Channel: AWGN

In [ ]:
def awgn_channel(sig_i, sig_q, snr_db, rng):
    power = np.mean(sig_i.astype(np.float64)**2 + sig_q.astype(np.float64)**2)
    noise_power = power / (10 ** (snr_db / 10))
    sigma = np.sqrt(noise_power / 2)
    ni = rng.normal(0, sigma, size=sig_i.shape)
    nq = rng.normal(0, sigma, size=sig_q.shape)
    # Quantize back to integer (W_2=40 domain)
    return np.round(sig_i + ni).astype(np.int64), np.round(sig_q + nq).astype(np.int64)


## 7. RX RRC (matched filter — `fir_rrc_rx` IP)

동일 계수. 출력 W_3=56.

In [ ]:
def fir_rrc_rx(x, coefs=RRC_COEFS_INT):
    return np.convolve(x.astype(np.int64), coefs, mode="full").astype(np.int64)


## 8. `axis_downsample_pick` (SPS× → 1×, `phase==OFFSET` pick)

Top-level 이 `OFFSET=0` 로 인스턴스화 → 인덱스 0, SPS, 2·SPS, ... 만 유지.


In [ ]:
def axis_downsample_pick(sig_i, sig_q, sps=SPS, offset=OFFSET):
    return sig_i[offset::sps], sig_q[offset::sps]


## 9. `preamble_correlator`

RTL 구조:
```
shift_i[0] = newest (rx[n]),  shift_i[L-1] = oldest (rx[n-L+1])
sum_i = Σ_k shift_i[k]·P_I[L-1-k] + shift_q[k]·P_Q[L-1-k]
sum_q = Σ_k shift_q[k]·P_I[L-1-k] − shift_i[k]·P_Q[L-1-k]
```
결국 수식은 **time-reversed reference 와의 linear convolution** 과 등가.
여기선 중첩 루프 대신 `np.convolve` 로 벡터화 (수치는 동일).


In [ ]:
def preamble_correlator(rx_i, rx_q):
    """Return (corr_i, corr_q) with valid values starting from index L-1."""
    # shift[k] = rx[n-k] → multiply with P[L-1-k]
    # Since shift[k] is a time-reversal of a window, Σ shift[k]*P[L-1-k]
    # = convolution of rx with P (non-reversed), taken at index n.
    L = PREAMBLE_LEN

    # np.convolve(rx, P)[L-1 : L-1+N] = Σ_{k} rx[n-k]*P[k]
    # We want Σ rx[n-k]*P[L-1-k] → convolve with time-reversed P.
    P_I_rev = PREAMBLE_I[::-1]
    P_Q_rev = PREAMBLE_Q[::-1]

    N = len(rx_i)
    ci = (np.convolve(rx_i, P_I_rev)[L-1:L-1+N]
        + np.convolve(rx_q, P_Q_rev)[L-1:L-1+N])
    cq = (np.convolve(rx_q, P_I_rev)[L-1:L-1+N]
        - np.convolve(rx_i, P_Q_rev)[L-1:L-1+N])

    # Before we've shifted in a full preamble, mask outputs to 0 (RTL sees 0s in shift reg)
    # (np.convolve already gives that naturally because rx is padded with nothing — the
    #  first L-1 samples are partial. Zero them to match RTL init behaviour exactly.)
    ci[:L-1] = 0
    cq[:L-1] = 0
    return ci, cq


## 10. `frame_sync_detector`

RTL 핵심:
- `mag_sq = corr_i*corr_i + corr_q*corr_q` (128-bit 연산)
- **비교는 `mag_sq[127:64] > THRESHOLD`** — 즉 `mag_sq >> 64` 의 상위부만 사용
- `sync_found` 는 1-cycle 지연 후 발생, `sync_index = sample_counter - 1`
- 한번 검출되면 32 cycle cooldown

Python 상에서 `mag_sq >> 64` 는 큰 정수 비트 시프트로 재현.


In [ ]:
def frame_sync_detector(corr_i, corr_q,
                          threshold=SYNC_THRESHOLD,
                          cooldown=COOLDOWN_CYCLES):
    """
    Returns:
      detections : list of (sync_index, sync_mag_upper64)
      mag_sq_upper64 : per-sample upper-64-bit magnitude (for plotting in log scale)
    """
    # Use Python arbitrary-precision int (ci, cq can exceed int64 so cast to object)
    ci = corr_i.astype(object)
    cq = corr_q.astype(object)
    mag_sq = (ci * ci + cq * cq)                 # full precision, Python int
    mag_upper = np.array([int(m) >> 64 for m in mag_sq], dtype=np.int64)

    detections = []
    cd = 0
    for n in range(len(mag_upper)):
        if cd > 0:
            cd -= 1
            continue
        if mag_upper[n] > threshold:
            detections.append((n, int(mag_upper[n])))
            cd = cooldown
    return detections, mag_upper


## 11. `qpsk_demodulator`

Hard decision by sign bit. RTL:
```
out_data[0] = in_i[W-1];   // I sign bit (MSB)
out_data[1] = in_q[W-1];   // Q sign bit
```
I≥0 → 0, I<0 → 1  (Q 동일).


In [ ]:
def qpsk_demodulator(rx_i, rx_q):
    """Returns 2-bit values per symbol (Q<<1 | I), matching out_data."""
    bi = (rx_i < 0).astype(np.int8)
    bq = (rx_q < 0).astype(np.int8)
    return ((bq << 1) | bi).astype(np.int8)


def pairs_to_bits(pairs):
    """Unpack 2-bit values back to a flat bit stream (LSB=I, MSB=Q per pair)."""
    n = len(pairs)
    bits = np.empty(n * 2, dtype=np.int8)
    bits[0::2] = pairs & 0x1         # I
    bits[1::2] = (pairs >> 1) & 0x1  # Q
    return bits


---
# Step-by-step interactive simulation

아래 셀들을 순서대로 실행하며 단계별 결과를 시각화합니다. 파라미터는 자유롭게 변경 가능.


In [ ]:
# ---- 실험 파라미터 ----
n_payload_syms = 200
snr_db         = 20
seed           = 42
rng = np.random.default_rng(seed)
print(f"n_payload_syms={n_payload_syms}, snr_db={snr_db}, seed={seed}")


### Step 1 — Bits → QPSK symbols

In [ ]:
tx_bits = rng.integers(0, 2, size=n_payload_syms * 2, dtype=np.int8)
tx_pairs = bits_to_pairs(tx_bits)
payload_i, payload_q = qpsk_modulator(tx_pairs)

plt.figure(figsize=(5, 5))
plt.scatter(payload_i, payload_q, s=30, alpha=0.5)
plt.title("TX Payload Constellation (±SCALE)")
plt.xlabel("I"); plt.ylabel("Q"); plt.grid(True); plt.axis("equal"); plt.show()

print("첫 8 심볼:", list(zip(payload_i[:8], payload_q[:8])))


### Step 2 — Frame build (preamble prepend)

In [ ]:
frame_i, frame_q = frame_builder(payload_i, payload_q)
print(f"Frame: {PREAMBLE_LEN} preamble + {n_payload_syms} payload = {len(frame_i)} syms")

fig, ax = plt.subplots(1, 1, figsize=(12, 2.5))
ax.stem(frame_i[:30], linefmt="C0-", markerfmt="C0o", label="I")
ax.stem(frame_q[:30], linefmt="C1-", markerfmt="C1x", label="Q")
ax.axvline(PREAMBLE_LEN - 0.5, color="r", linestyle="--", label="payload start")
ax.set_title("Frame symbols (first 30)")
ax.set_xlabel("symbol index"); ax.legend(); ax.grid(True); plt.show()


### Step 3 — Upsample (zero insertion ×SPS)

In [ ]:
up_i, up_q = axis_upsample_zeros(frame_i, frame_q)
print(f"Upsampled length: {len(up_i)} samples (= {len(frame_i)} × {SPS})")

fig, ax = plt.subplots(1, 1, figsize=(12, 3))
n_show = 20 * SPS
ax.stem(up_i[:n_show], linefmt="C0-", markerfmt="C0o")
ax.set_title("Upsample zeros — I channel (first 20 symbols × SPS)")
ax.set_xlabel("sample"); ax.grid(True); plt.show()


### Step 4 — TX RRC pulse shaping

In [ ]:
tx_i = fir_rrc(up_i)
tx_q = fir_rrc(up_q)
print(f"TX samples: {len(tx_i)}  (range: [{tx_i.min():,}, {tx_i.max():,}])")

fig, ax = plt.subplots(1, 1, figsize=(12, 3))
n_show = 30 * SPS
ax.plot(tx_i[:n_show], label="I")
ax.plot(tx_q[:n_show], label="Q", alpha=0.7)
ax.set_title("TX waveform after RRC shaping")
ax.set_xlabel("sample"); ax.legend(); ax.grid(True); plt.show()


### Step 5 — AWGN channel

In [ ]:
rx_i, rx_q = awgn_channel(tx_i, tx_q, snr_db, rng)

fig, ax = plt.subplots(1, 1, figsize=(12, 3))
n_show = 30 * SPS
ax.plot(rx_i[:n_show], label="I", alpha=0.7)
ax.plot(rx_q[:n_show], label="Q", alpha=0.5)
ax.set_title(f"RX waveform after AWGN (SNR={snr_db} dB)")
ax.set_xlabel("sample"); ax.legend(); ax.grid(True); plt.show()


### Step 6 — RX matched filter + downsample

In [ ]:
rxf_i = fir_rrc_rx(rx_i)
rxf_q = fir_rrc_rx(rx_q)

rxd_i, rxd_q = axis_downsample_pick(rxf_i, rxf_q)
print(f"MF out: {len(rxf_i)}, downsampled (OFFSET={OFFSET}): {len(rxd_i)} symbols")

fig, ax = plt.subplots(1, 1, figsize=(12, 3))
n_show = 30 * SPS
ax.plot(rxf_i[:n_show], label="I", alpha=0.7)
ds_idx = np.arange(OFFSET, n_show, SPS)
ax.plot(ds_idx, rxf_i[ds_idx], "ro", markersize=4, label="sample pts (offset=0)")
ax.set_title("MF output + downsample points")
ax.set_xlabel("sample"); ax.legend(); ax.grid(True); plt.show()


### Step 7 — Preamble correlation (matched filter for preamble)

In [ ]:
corr_i, corr_q = preamble_correlator(rxd_i, rxd_q)

# Full-precision magnitude squared
mag_sq_full = corr_i.astype(object) ** 2 + corr_q.astype(object) ** 2
# Upper 64 bits (what the hardware compares)
mag_upper = np.array([int(m) >> 64 for m in mag_sq_full], dtype=np.int64)

peak = int(mag_upper.max())
print(f"peak mag_sq[127:64] = {peak:,}")
print(f"SYNC_THRESHOLD      = {SYNC_THRESHOLD:,}")
print(f"peak > threshold?     {peak > SYNC_THRESHOLD}")

fig, axes = plt.subplots(2, 1, figsize=(12, 6))
axes[0].plot(mag_upper)
axes[0].axhline(SYNC_THRESHOLD, color="r", linestyle="--", label=f"threshold={SYNC_THRESHOLD:.0e}")
axes[0].set_title("|R|² upper-64-bit (what frame_sync_detector compares)")
axes[0].set_xlabel("symbol index"); axes[0].set_yscale("symlog")
axes[0].legend(); axes[0].grid(True)

# For intuition: also plot full |R|² in log scale
mag_full = np.array([int(m) for m in mag_sq_full], dtype=float)
axes[1].plot(mag_full)
axes[1].set_title("|R|² full precision (reference)")
axes[1].set_xlabel("symbol index"); axes[1].set_yscale("log"); axes[1].grid(True)
plt.tight_layout(); plt.show()


### Step 8 — Frame sync detection

In [ ]:
detections, mag_upper = frame_sync_detector(corr_i, corr_q)
print(f"detections ({len(detections)}):")
for idx, m in detections:
    print(f"  sym_idx={idx}, mag_upper={m:,}")

fig, ax = plt.subplots(1, 1, figsize=(12, 3))
ax.plot(mag_upper)
ax.axhline(SYNC_THRESHOLD, color="r", linestyle="--", label=f"threshold={SYNC_THRESHOLD:.0e}")
for idx, _ in detections:
    ax.axvline(idx, color="g", linestyle="--", alpha=0.5)
ax.set_yscale("symlog")
ax.set_title("Detections (green = sync_index)"); ax.set_xlabel("symbol index")
ax.legend(); ax.grid(True); plt.show()


### Step 9 — Demodulate + BER

RTL 상 `qpsk_demodulator` 는 `rx_dn_i/q` 전체에 대해 항상 동작하지만, sync 를 기반으로 payload 구간만 잘라 BER 계산.


In [ ]:
rx_pairs = qpsk_demodulator(rxd_i, rxd_q)

if detections:
    sync_idx = detections[0][0]
    # Correlator peak occurs at the LAST preamble symbol (shift register fully filled)
    # → payload starts at sync_idx + 1
    ps = sync_idx + 1
    pe = ps + n_payload_syms

    if pe <= len(rx_pairs):
        rx_bits = pairs_to_bits(rx_pairs[ps:pe])
        ber = np.mean(rx_bits != tx_bits)
        print(f"BER = {ber:.6e}  (SNR={snr_db} dB, {n_payload_syms} payload syms)")
        print(f"errors: {int(np.sum(rx_bits != tx_bits))} / {len(tx_bits)}")

        plt.figure(figsize=(5, 5))
        plt.scatter(rxd_i[ps:pe], rxd_q[ps:pe], s=10, alpha=0.5)
        plt.title(f"RX Payload Constellation  BER={ber:.2e}")
        plt.xlabel("I"); plt.ylabel("Q"); plt.grid(True); plt.axis("equal"); plt.show()
    else:
        print(f"payload range ({ps}:{pe}) exceeds rxd length {len(rx_pairs)} — 심볼 더 길게 생성 필요")
else:
    print("No sync detected — SYNC_THRESHOLD 낮추거나 SNR 올려보세요.")


### Step 10 — Eye diagram (MF output, I channel)

Symbol boundary 에 맞춰 2-심볼 윈도우 반복 overlay.


In [ ]:
eye_start = 2 * RRC_DELAY   # skip filter transient
L = 2 * SPS
n_traces = min(200, (len(rxf_i) - eye_start) // L)
t_eye = np.arange(L) / SPS

plt.figure(figsize=(10, 4))
for k in range(n_traces):
    s = eye_start + k * L
    plt.plot(t_eye, rxf_i[s:s+L], color="C0", alpha=0.1)
plt.title(f"Eye diagram — I (SNR={snr_db} dB, {n_traces} traces)")
plt.xlabel("time (symbols)"); plt.grid(True); plt.show()


---
## Appendix — Bit-accuracy sanity check vs RTL testbench

`src/tb/tv_*.hex` 테스트벡터가 있으면 그걸 로드해서 이 파이썬 모델이 RTL 과 같은 숫자를 뽑는지 체크.


In [ ]:
import os
TB_DIR = "/home/hong/workspace_claude/modem/scm/src/tb"

def load_hex(path, signed=True, width=16):
    """Load tv_*.hex produced by testbench."""
    vals = []
    with open(path) as f:
        for line in f:
            s = line.strip()
            if not s: continue
            v = int(s, 16)
            if signed and v >= (1 << (width - 1)):
                v -= (1 << width)
            vals.append(v)
    return np.array(vals, dtype=np.int64)

# Example: compare TX modulator output
if os.path.exists(f"{TB_DIR}/tv_tx_bits.hex") and os.path.exists(f"{TB_DIR}/tv_mod_i.hex"):
    tv_bits = load_hex(f"{TB_DIR}/tv_tx_bits.hex", signed=False, width=2)
    tv_mod_i = load_hex(f"{TB_DIR}/tv_mod_i.hex", signed=True, width=16)
    tv_mod_q = load_hex(f"{TB_DIR}/tv_mod_q.hex", signed=True, width=16)

    mod_i, mod_q = qpsk_modulator(tv_bits)
    n = min(len(mod_i), len(tv_mod_i))
    match_i = np.array_equal(mod_i[:n], tv_mod_i[:n])
    match_q = np.array_equal(mod_q[:n], tv_mod_q[:n])
    print(f"qpsk_modulator match vs TV: I={match_i}, Q={match_q}, samples={n}")
else:
    print("testbench hex files 없음 — 위 경로 확인")
